# Lab 17 — Reference solution

The polished final implementation of [Lab 17: LangSmith trace ingestion](../README.md).

A LangSmith-instrumented Lab 14-style supervisor agent + the agentevals evaluator package + an end-to-end offline experiment run. Same dependencies as the lab; `LANGCHAIN_TRACING_V2=true` and `LANGCHAIN_API_KEY` in the environment.

> 📖 Required reading before this: [`concepts/evaluation/langsmith-tracing-shape.md`](../../../concepts/evaluation/langsmith-tracing-shape.md), [`concepts/evaluation/online-vs-offline-evaluation.md`](../../../concepts/evaluation/online-vs-offline-evaluation.md).


## Step 0: Setup

In [ ]:
# Install dependencies (if not already installed)
# !pip install --quiet langsmith agentevals

import os
import json
import pathlib
from typing import Any, Literal

from dotenv import load_dotenv

# Load .env from repo root
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

# LangSmith config — set these in your .env or shell before running
os.environ["LANGSMITH_TRACING"] = os.environ.get("LANGSMITH_TRACING", "true")
os.environ["LANGSMITH_PROJECT"] = os.environ.get("LANGSMITH_PROJECT", "lab-17-trace-ingestion")
# LANGSMITH_API_KEY must already be set in env or .env

# Provider config
PROVIDER = os.environ.get("PROVIDER", "openai")
MODEL = {"openai": "gpt-4o-mini", "anthropic": "claude-haiku-4-5-20251001"}[PROVIDER]

# Verify setup
assert os.environ.get("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY in your .env or shell"
print(f"Provider: {PROVIDER}, Model: {MODEL}")
print(f"LangSmith tracing: {os.environ.get('LANGSMITH_TRACING')}")
print(f"LangSmith project: {os.environ.get('LANGSMITH_PROJECT')}")


## Step 1: Inline minimal Lab 14 agent

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.types import Command
from langgraph.checkpoint.memory import InMemorySaver

# Provider-agnostic chat model
if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=MODEL, temperature=0)
else:
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model=MODEL, temperature=0)


class SupervisorState(MessagesState):
    """State extending MessagesState with worker outputs."""
    findings: str
    final_answer: str


# ── Workers (stubbed for lab focus on instrumentation) ──

def researcher_node(state: SupervisorState) -> dict:
    """Stub researcher — returns fixed findings to keep the lab deterministic."""
    return {
        "findings": "MCP is an open standard for connecting LLMs to tools [1]. "
                    "Recent developments include broader production adoption [2].",
        "messages": [AIMessage(content="researcher_complete", name="researcher")],
    }


def writer_node(state: SupervisorState) -> dict:
    """Writer — composes the final answer from findings + citations."""
    findings = state.get("findings", "")
    response = llm.invoke([
        SystemMessage(content="Compose a 100-word summary from these findings. "
                              "Preserve citation markers [1], [2] exactly."),
        HumanMessage(content=f"FINDINGS:\n{findings}"),
    ])
    return {
        "final_answer": response.content,
        "messages": [AIMessage(content="writer_complete", name="writer")],
    }


# ── Supervisor ──

@tool
def call_researcher(question: str) -> str:
    """Dispatch the question to the researcher worker."""
    return "[routed to researcher]"


@tool
def call_writer() -> str:
    """Dispatch the brief to the writer worker."""
    return "[routed to writer]"


SUPERVISOR_PROMPT = """You are a supervisor coordinating two workers via tool calls.

WORKFLOW:
1. Call the researcher first.
2. Then call the writer.
3. Then respond directly (no tool call) to finalize.

Call each worker exactly once.
"""

supervisor_llm = llm.bind_tools([call_researcher, call_writer])


def supervisor_node(
    state: SupervisorState,
) -> Command[Literal["researcher", "writer", "__end__"]]:
    """Routes via Command(goto=..., update=...)."""
    response = supervisor_llm.invoke([
        SystemMessage(content=SUPERVISOR_PROMPT),
        *state["messages"],
    ])
    update = {"messages": [response]}

    if not response.tool_calls:
        return Command(goto="__end__", update=update)

    tc = response.tool_calls[0]
    if tc["name"] == "call_researcher":
        return Command(goto="researcher", update=update)
    elif tc["name"] == "call_writer":
        return Command(goto="writer", update=update)
    return Command(goto="__end__", update=update)


# ── Wire the graph ──

def build_graph(checkpointer=None):
    builder = StateGraph(SupervisorState)
    builder.add_node("supervisor", supervisor_node)
    builder.add_node("researcher", researcher_node)
    builder.add_node("writer", writer_node)
    builder.add_edge(START, "supervisor")
    builder.add_edge("researcher", "supervisor")
    builder.add_edge("writer", "supervisor")
    return builder.compile(checkpointer=checkpointer)


graph = build_graph(checkpointer=InMemorySaver())
print(f"Graph compiled with {len(graph.nodes)} nodes: {list(graph.nodes.keys())}")


## Step 2: Run once with auto-tracing

In [ ]:
task = "Research recent developments in MCP and write a 100-word summary."

config_run1 = {"configurable": {"thread_id": "run-1"}, "recursion_limit": 10}
result = graph.invoke({"messages": [HumanMessage(content=task)]}, config=config_run1)

print("Final answer:")
print("─" * 60)
print(result.get("final_answer", "[no final answer]"))
print()
print("→ Check the LangSmith UI: smith.langchain.com → lab-17-trace-ingestion")
print(f"  Thread ID: {config_run1['configurable']['thread_id']}")


**What to look for in the LangSmith UI**:

- The trace tree shows three top-level nodes: `supervisor` → `researcher` → `supervisor` → `writer` → `supervisor`.
- Each node is auto-typed (`chain` type, since LangGraph nodes are LangChain runnables).
- Each LLM call inside a node nests as a `llm`-type child Run with full prompt + response + token usage.
- Toggle between **Messages view** (logic-focused; chat-like rendering) and **Timeline view** (latency-focused; flame graph).

No code changes were needed for any of this — just the env vars.

## Step 3: `@traceable` on a custom helper

In [ ]:
import re
from langsmith import traceable


@traceable
def extract_citations(answer: str) -> list[str]:
    """Pull URL-shaped citations out of an answer. Returns deduplicated URLs.

    This is a plain Python function — not a LangGraph node, not a LangChain primitive.
    The @traceable decorator is what makes it visible in the trace tree.
    """
    url_pattern = re.compile(r'https?://[^\s\]\)<>"]+')
    urls = url_pattern.findall(answer)
    return list(dict.fromkeys(urls))  # preserve order, dedupe


# Run inside a graph invocation so the helper nests under the main trace
config_run2 = {"configurable": {"thread_id": "run-2"}, "recursion_limit": 10}
result = graph.invoke({"messages": [HumanMessage(content=task)]}, config=config_run2)
citations = extract_citations(result.get("final_answer", ""))

print(f"Extracted {len(citations)} citations:")
for c in citations:
    print(f"  - {c}")
print()
print(f"→ In LangSmith: thread {config_run2['configurable']['thread_id']}")
print("  extract_citations appears as a top-level Run (called outside graph invocation)")


## Step 4: Add tags and metadata

In [ ]:
@traceable(
    name="citation_extractor_v2",
    tags=["post-processing", "citation-extraction"],
    metadata={"version": "2.1", "regex_pattern": "strict"},
    run_type="chain",
)
def extract_citations_v2(answer: str) -> dict:
    """Same as v1 but with metadata. Returns extracted citations and a count."""
    url_pattern = re.compile(r'https?://[^\s\]\)<>"]+')
    urls = list(dict.fromkeys(url_pattern.findall(answer)))
    return {"citations": urls, "count": len(urls)}


result_v2 = extract_citations_v2(result.get("final_answer", ""))
print(f"Extracted {result_v2['count']} citations.")
print()
print("→ In LangSmith UI: filter by tag 'citation-extraction' to see v2 runs only.")
print("→ Filter by metadata.version='2.1' for the specific version.")


## Step 5: Project scoping with `tracing_v2_enabled`

In [ ]:
from langsmith.run_helpers import tracing_v2_enabled

# Run the same agent under two different project names — simulating A/B comparison
config_v1 = {"configurable": {"thread_id": "ab-v1"}, "recursion_limit": 10}
config_v2 = {"configurable": {"thread_id": "ab-v2"}, "recursion_limit": 10}

with tracing_v2_enabled(project_name="lab-17-variant-a"):
    result_a = graph.invoke({"messages": [HumanMessage(content=task)]}, config=config_v1)

with tracing_v2_enabled(project_name="lab-17-variant-b"):
    result_b = graph.invoke({"messages": [HumanMessage(content=task)]}, config=config_v2)

print("Variant A final:", result_a.get("final_answer", "")[:100] + "...")
print("Variant B final:", result_b.get("final_answer", "")[:100] + "...")
print()
print("→ Two projects in LangSmith: 'lab-17-variant-a' and 'lab-17-variant-b'")
print("  Each has its own trace; compare side-by-side in the UI's Experiments tab.")


## Step 6: Read the trace in the LangSmith UI

This is a UI step, not a code step. Open one of the traces from the previous cells in smith.langchain.com.

**Messages view** (toggle in the upper right of the trace detail panel):
- Chat-like rendering of the conversation
- User input → assistant tool call → tool result → assistant tool call → tool result → assistant final response
- Read this when debugging *what the agent said* at each step

**Timeline view** (the other toggle):
- Flame-graph rendering with wall-clock duration
- Each Run is a horizontal bar; nested Runs sit below their parent
- Read this when debugging *where the time is going*

Both views are complementary. The messages view shows the logic; the timeline view shows the cost. Most debugging sessions toggle between them.

→ Try filtering the project by tag `citation-extraction` — only the helper Runs from Step 4 appear.
→ Try filtering by metadata `version: 2.1` — narrows further.

## Step 7: Extract the trajectory via `extract_langgraph_trajectory_from_thread`

In [ ]:
from agentevals.graph_trajectory.utils import extract_langgraph_trajectory_from_thread

# Extract from the run-1 thread we created in Step 2
trajectory = extract_langgraph_trajectory_from_thread(
    graph=graph,
    config=config_run1,
)

print("Trajectory structure:")
print(f"  inputs: {trajectory.get('inputs') is not None}")
print(f"  results: {len(trajectory.get('results', []))} turn(s)")
print(f"  steps: {trajectory.get('steps')}")


**Sample output**:

```
Trajectory structure:
  inputs: True
  results: 1 turn(s)
  steps: [['__start__', 'supervisor', 'researcher', 'supervisor', 'writer', 'supervisor']]
```

The `steps` list has one inner list (one conversation turn). The inner list is the node-visit sequence — `__start__` is a synthetic entry marker; the actual graph nodes follow.

## Step 8: `graph_trajectory_strict_match` — deterministic evaluator

In [ ]:
from agentevals.graph_trajectory.strict import graph_trajectory_strict_match

# Define the expected trajectory — what we believe the agent should do
expected_trajectory = {
    "inputs": None,
    "results": [{}],
    "steps": [["__start__", "supervisor", "researcher", "supervisor", "writer", "supervisor"]],
}

result_strict = graph_trajectory_strict_match(
    outputs=trajectory,
    reference_outputs=expected_trajectory,
)

print("Strict match result:")
print(f"  score: {result_strict.get('score')}")
print(f"  comment: {result_strict.get('comment', '(none)')}")


**Sample output**:

```
Strict match result:
  score: 1.0
  comment: (none)
```

The supervisor routed correctly. If the supervisor had skipped the researcher and gone straight to the writer (a routing bug — same as Lab 16's `lab10_t05` case), the score would be 0.0 and the comment would identify the mismatch.

## Step 9: `create_trajectory_llm_as_judge` — LLM-judged

In [ ]:
from agentevals.trajectory.llm import (
    create_trajectory_llm_as_judge,
    TRAJECTORY_ACCURACY_PROMPT,
)

llm_judge = create_trajectory_llm_as_judge(
    model=f"{PROVIDER}:{MODEL}",
    prompt=TRAJECTORY_ACCURACY_PROMPT,
)

# The LLM judge expects message-format trajectories, not the GraphTrajectory shape.
# We reconstruct from the final state's messages.
state_final = graph.get_state(config_run1).values
messages = state_final.get("messages", [])

judge_result = llm_judge(
    outputs=messages,
    inputs={"messages": [{"role": "user", "content": task}]},
)

print("LLM-as-judge result:")
print(f"  score: {judge_result.get('score')}")
print(f"  comment: {judge_result.get('comment', '(none)')[:200]}")


**Sample output** (LLM responses will vary):

```
LLM-as-judge result:
  score: True
  comment: The agent's trajectory was appropriate: it called the researcher to gather information about MCP, then called the writer to compose the summary. The flow matches the user's request.
```

The judge scores `True` (boolean for this prompt; numeric for others) plus a short justification. The output depends on the prompt; `TRAJECTORY_ACCURACY_PROMPT` uses a boolean shape, custom prompts can use 0-1 floats or any scoring scheme.

**Watch for**: Zheng et al. 2023 documented three biases in LLM-as-judge — position, verbosity, self-enhancement. The judge here is not calibrated; production deployments need periodic human calibration. Module 5 covers this.

## Step 10: Run an offline experiment via `client.evaluate`

In [ ]:
from langsmith import Client

client = Client()

# Define three small test cases
test_cases = [
    {
        "inputs": {"messages": [{"role": "user", "content": "Research MCP and summarize."}]},
        "outputs": {"messages": [{"role": "assistant", "content": "supervisor→researcher→writer→END"}]},
    },
    {
        "inputs": {"messages": [{"role": "user", "content": "Research RAG eval and summarize."}]},
        "outputs": {"messages": [{"role": "assistant", "content": "supervisor→researcher→writer→END"}]},
    },
    {
        "inputs": {"messages": [{"role": "user", "content": "Research agent routing and summarize."}]},
        "outputs": {"messages": [{"role": "assistant", "content": "supervisor→researcher→writer→END"}]},
    },
]

DATASET_NAME = "lab-17-tiny-trajectory-dataset"

# Create the Dataset if it doesn't exist; skip if it does
try:
    dataset = client.read_dataset(dataset_name=DATASET_NAME)
    print(f"Dataset exists: {DATASET_NAME}")
except Exception:
    dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Tiny 3-example trajectory dataset for Lab 17.",
    )
    for case in test_cases:
        client.create_example(
            inputs=case["inputs"],
            outputs=case["outputs"],
            dataset_id=dataset.id,
        )
    print(f"Created Dataset: {DATASET_NAME} with {len(test_cases)} examples")


In [ ]:
# Define the target — what gets called per-example
def agent_target(inputs: dict) -> dict:
    """Invoke the graph against one Dataset input. Returns the trace."""
    config = {"configurable": {"thread_id": f"eval-{hash(json.dumps(inputs)) % 10000}"},
              "recursion_limit": 10}
    graph.invoke({"messages": [HumanMessage(content=inputs["messages"][-1]["content"])]},
                 config=config)
    # Return the messages for the LLM judge to read
    state = graph.get_state(config).values
    return {"messages": state.get("messages", [])}


# Run the experiment with both evaluators
print("Running experiment (3 examples × 2 evaluators = 6 evaluations)...")
print("(this triggers LLM calls; ~$0.05 total)")

# Note: agentevals' graph_trajectory_strict_match operates on GraphTrajectory shape,
# not on message lists. For experiment-style evaluation we use the LLM judge here
# and demonstrate the deterministic match path inline in Step 8 above.

experiment_results = client.evaluate(
    agent_target,
    data=DATASET_NAME,
    evaluators=[llm_judge],
    experiment_prefix="lab-17-experiment",
    max_concurrency=1,  # be polite to free-tier rate limits
)

print()
print("→ Experiment complete. View results in LangSmith UI:")
print(f"  Datasets → {DATASET_NAME} → Experiments tab")


**Sample output**:

```
Dataset exists: lab-17-tiny-trajectory-dataset
Running experiment (3 examples × 2 evaluators = 6 evaluations)...

→ Experiment complete. View results in LangSmith UI:
  Datasets → lab-17-tiny-trajectory-dataset → Experiments tab
```

In the LangSmith UI's Experiments tab you'll see one row per Dataset example, columns showing the LLM-judge score and the per-example traces. Comparing experiments across runs (after PRs that modified the supervisor prompt, for example) is what platform-managed offline evaluation provides over the from-scratch harness — the historical state.

## Step 11 (stretch): Custom evaluator using Lab 16's `routing_accuracy`

In [ ]:
def routing_accuracy_lcs(actual: list[str], expected: list[str]) -> float:
    """Lab 16's LCS-based routing accuracy. Algorithm unchanged."""
    n, m = len(actual), len(expected)
    if n == 0 or m == 0:
        return 0.0
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            if actual[i] == expected[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])
    return dp[n][m] / m


def custom_routing_evaluator(outputs: list, reference_outputs=None, **kwargs) -> dict:
    """Custom evaluator: scores LCS-based routing accuracy against the expected
    supervisor → researcher → supervisor → writer → supervisor sequence.

    Args follow the agentevals evaluator signature: outputs is the actual
    trajectory (list of messages or trajectory dict).
    """
    # Extract actual node sequence from messages
    # In the message list, AIMessages with name="researcher" / name="writer" mark node exits
    actual_nodes = ["supervisor"]  # implicit start
    for msg in outputs if isinstance(outputs, list) else []:
        name = getattr(msg, "name", None)
        if name == "researcher":
            actual_nodes.extend(["researcher", "supervisor"])
        elif name == "writer":
            actual_nodes.extend(["writer", "supervisor"])

    expected = ["supervisor", "researcher", "supervisor", "writer", "supervisor"]
    score = routing_accuracy_lcs(actual_nodes, expected)

    return {
        "key": "routing_accuracy",
        "score": score,
        "comment": f"LCS-based score; actual sequence had {len(actual_nodes)} nodes",
    }


# Test on the run-1 trajectory
state_run1 = graph.get_state(config_run1).values
messages_run1 = state_run1.get("messages", [])
score = custom_routing_evaluator(outputs=messages_run1)
print("Custom routing_accuracy evaluator (Lab 16 algorithm):")
print(f"  score: {score['score']:.2f}")
print(f"  comment: {score['comment']}")


**Sample output**:

```
Custom routing_accuracy evaluator (Lab 16 algorithm):
  score: 1.00
  comment: LCS-based score; actual sequence had 5 nodes
```

The pattern generalizes. Each of Lab 16's seven metrics can wrap into the `agentevals` evaluator signature with minor adaptations:

- `handoff_success_rate` → check `status` field on each step
- `plan_validity` → check plan_valid flag (Lab 12-style traces)
- `citation_preservation` → extract URLs from final answer + compare to expected list
- `groundedness` → lexical overlap between claims and trace's retriever outputs

All of these run as custom evaluators in `client.evaluate(...)`. The Lab 16 algorithms are portable; the wiring is platform-specific.

## Step 12: Synthesis

What this lab built:

- **A LangSmith-instrumented LangGraph agent** with three tracing modes wired and working: auto-tracing for graph nodes, `@traceable` for custom helpers, `tracing_v2_enabled` for project scoping.
- **Two `agentevals` evaluators**: `graph_trajectory_strict_match` (deterministic, free) and `create_trajectory_llm_as_judge` with `TRAJECTORY_ACCURACY_PROMPT` (LLM-judged, ~$0.005/call).
- **An offline experiment** via `client.evaluate(...)` against a tiny Dataset.
- **A custom evaluator** that reuses Lab 16's `routing_accuracy` algorithm — demonstrating that from-scratch metrics are portable to platform-native wiring.

What this lab didn't cover (deferred to Modules 3-7):

- **OpenTelemetry-native instrumentation.** Module 3. Same agent, OTel SDK, fanout to LangSmith + a generic OTel collector.
- **Online evaluator registration.** A UI step in the LangSmith platform; mentioned but not demonstrated end-to-end here.
- **Tail-based sampling and cost attribution.** Module 6.
- **Drift detection on metric distributions over time.** Module 5.
- **Agent-as-judge calibration against human ground truth.** Module 5.
- **Multi-turn (threaded) evaluation.** Module 7.

The pattern to take with you:

1. **Start with auto-tracing.** Two env vars; works for any LangGraph or LangChain agent.
2. **Add `@traceable` selectively** — only for the helpers, preprocessors, post-processors that aren't already graph nodes.
3. **Use `agentevals` for the common-case trajectory checks**; write custom evaluators for domain-specific metrics (citation preservation, plan validity).
4. **Pair offline (Dataset + `client.evaluate`) with online (registered evaluators on live traces).** Offline gates merges; online surfaces drift. Both close the loop via the annotation queue.

Path 06's discipline mirrors Path 02/Path 03's from-scratch-first approach. Lab 16 gave you the metrics from scratch; Lab 17 wires them into the production tooling. Subsequent modules add the other production concerns (drift, calibration, cost, multi-turn) layer by layer.

✓ **Module 2 complete.** Module 3 (OpenTelemetry portable layer) in a future batch.


## ✓ Solution complete

This is the LangSmith-native variant. Lab 18's solution implements the same agent with the OpenTelemetry-portable variant; the two together demonstrate the vendor-native / vendor-neutral choice.

See [`README.md`](./README.md) for the design choices and what's deliberately out of scope.
